# Training notebook

## Pre-processing

In [ ]:
from multipride.preprocessing import load_tokenizer, tokenize_batch
from multipride.evaluation import compute_metrics
from multipride.data_utils import load_split
from multipride.training_utils import run_hyperparameter_search
from transformers import (
    AutoModelForSequenceClassification,
)
import torch
import wandb

In [ ]:
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
print("Device name:", torch.cuda.get_device_name(0))
print("Current device:", torch.cuda.current_device())

# quick tensor test
x = torch.rand(3, 3).to("cuda")
print("Tensor on GPU:", x.device)

In [ ]:
RUN_NAME = "third_local_run"
WANDB_PROJECT_NAME = "transformer-fine-tuning"
MODEL_CHECKPOINT = "Musixmatch/umberto-commoncrawl-cased-v1"

In [ ]:
# load and split the dataset
data_dict = load_split(
    "csv",
    "../data/processed/clean_data.csv",
    True,
    False,
    ["id", "text", "label"],
    0.15,
    42,
)
print(data_dict)

In [ ]:
tokenizer = load_tokenizer(MODEL_CHECKPOINT)

tokenized_datasets = data_dict.map(
    lambda batch: tokenize_batch(batch, tokenizer), batched=True
)

In [ ]:
train_set = tokenized_datasets["train"]

val_set = tokenized_datasets["test"]

In [ ]:
print(train_set["label"][:10])
print(val_set["label"][:10])

In [ ]:
wandb.login()

In [ ]:
def model_init(trial):
    # Ensure a fresh model is loaded for each trial
    return AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT)

In [ ]:
def optuna_hp_space(trial):
    return {
        # CRITICAL HYPERPARAMETERS TO TUNE (Phase 1)
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True),
        "per_device_train_batch_size": trial.suggest_categorical(
            "per_device_train_batch_size", [16, 32]
        ),
        # REGULARIZATION (Phase 1)
        # Note: We keep the number of epochs fixed and rely on Early Stopping
        "weight_decay": trial.suggest_float("weight_decay", 0.01, 0.1, log=True),
        "num_train_epochs": trial.suggest_int(
            "num_train_epochs", 3, 5
        ),  # Let's set a range but rely on Early Stopping
        # OTHER OPTIONAL BUT USEFUL SETTINGS
        "seed": trial.suggest_int("seed", 42, 100),
        "warmup_steps": trial.suggest_int("warmup_steps", 0, 500),
    }

In [ ]:
def compute_objective(metrics):
    """Returns the objective metric to maximize (Macro F1)."""
    val = metrics.get("f1") or metrics.get("eval_f1")
    if val is None:
        raise KeyError(
            f"Objective metric 'f1' not found in metrics: {list(metrics.keys())}"
        )
    return val

In [ ]:
best_run = run_hyperparameter_search(
    train_set,
    val_set,
    WANDB_PROJECT_NAME,
    tokenizer,
    optuna_hp_space,
    compute_metrics,
    model_init,
    compute_objective,
)